# Autonomous Sumo Robot Digital Twin - Stage 1: Data Gathering
## Sim-to-Real Telemetry Generation via 2D Differential Kinematics and Multi-Sensor Raycasting

This research notebook executes the deterministic 2D Digital Twin simulation environment to generate high-volume match telemetry for autonomous sumo robots (Kit 1kg and Mega 3kg classes).

### Objectives:
1. Initialize competition arena parameters and robot kinematic profiles.
2. Execute batch simulations (1,000 matches) across 5 benchmark adversary FSM strategies.
3. Capture raw multi-sensor telemetry at a 50ms tick rate ($\Delta t = 0.05$ s).
4. Export raw telemetry logs to `data/raw/` for downstream ETL and ML modeling.

In [2]:
from IPython.display import display, Markdown, HTML
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Ensure project root is in Python path
project_root = Path.cwd() if (Path.cwd() / 'config').exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.config import KIT_1KG_CONFIG, MEGA_3KG_CONFIG, RAW_DATA_DIR, PROJECT_ROOT
from environment.sumo_env import SumoEnvironment, run_simulation_match
from environment.opponents import (
    AggressiveCharger,
    DefensiveSweeper,
    RandomFlanker,
    BaitAndSwitch,
    JuggernautPush,
    RandomMixOpponent,
    OPPONENT_REGISTRY,
)

print('Libraries loaded successfully.')
print(f'Raw data output directory: {RAW_DATA_DIR}')


/Users/connorshyan/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/gs/psjxmgm545l6rznhpyskjqbc0000gn/T/matplotlib-spc1utwp because there was an issue with the default path ({configdir}); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.
Libraries loaded successfully.
Raw data output directory: /Users/connorshyan/UM/SumoDigitalTwin/data/raw


### 1. Batch Simulation Generator Function

In [4]:
def generate_batch_telemetry(
    config,
    adversary_name: str = 'RANDOM_MIX',
    num_matches: int = 1000,
    output_filename: str = 'telemetry_raw.parquet',
    formation_a: str = 'RANDOM_MIX',
    formation_b: str = 'RANDOM_MIX',
    seed: int = 42
) -> pd.DataFrame:
    """Simulate N matches and return aggregated raw telemetry DataFrame."""
    env = SumoEnvironment(config=config, seed=seed)
    
    # Bot A uses our candidate strategy (RandomMix or AggressiveCharger)
    bot_a_policy = RandomMixOpponent(config)
    
    # Bot B adversary
    adversary_cls = OPPONENT_REGISTRY.get(adversary_name, RandomMixOpponent)
    bot_b_policy = adversary_cls(config)
    
    all_records = []
    match_summaries = []
    
    print(f'[Simulating {num_matches} matches] Class: {config.class_name} | Adversary: {adversary_name} | Formations: {formation_a} vs {formation_b}...')
    for m_idx in tqdm(range(num_matches), desc='Simulating Matches'):
        match_id = f'M_{m_idx+1:05d}'
        records = run_simulation_match(
            env,
            bot_a_policy,
            bot_b_policy,
            match_id=match_id,
            formation_a=formation_a,
            formation_b=formation_b,
        )
        all_records.extend(records)
        
        # Last record gives final match status
        final_status = records[-1]['Match_Status']
        duration_ticks = records[-1]['Tick']
        match_summaries.append({
            'Match_ID': match_id,
            'Status': final_status,
            'Duration_Ticks': duration_ticks,
            'Duration_s': duration_ticks * config.dt
        })
        
    df_raw = pd.DataFrame(all_records)
    df_summary = pd.DataFrame(match_summaries)
    
    # Save raw Parquet
    out_path = RAW_DATA_DIR / output_filename
    df_raw.to_parquet(out_path, index=False, engine="pyarrow")
    print(f'Saved {len(df_raw):,} raw ticks across {num_matches} matches to {out_path}')
    
    # Print Outcome Distribution
    print('\n--- Match Outcome Summary ---')
    print(df_summary['Status'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')
    print(f'Mean Match Duration: {df_summary["Duration_s"].mean():.2f} s (+/- {df_summary["Duration_s"].std():.2f} s)')
    
    return df_raw


### 2. Generate 1 kg Kit Sumo Telemetry (1,000 Matches)

In [6]:
df_kit_raw = generate_batch_telemetry(
    config=KIT_1KG_CONFIG,
    adversary_name="RANDOM_MIX",
    num_matches=50,
    output_filename="telemetry_kit_1kg_raw.parquet",
    seed=42
)
df_kit_raw.head()

[Simulating 50 matches] Class: KIT_1KG | Adversary: RANDOM_MIX | Formations: RANDOM_MIX vs RANDOM_MIX...
Simulating Matches: 100%|##########| 50/50 [00:12<00:00,  4.06it/s]
Saved 114,060 raw ticks across 50 matches to /Users/connorshyan/UM/SumoDigitalTwin/data/raw/telemetry_kit_1kg_raw.parquet

--- Match Outcome Summary ---
Status
BOT_B_WIN    36.0 %
BOT_A_WIN    34.0 %
DRAW         30.0 %
Name: proportion, dtype: str
Mean Match Duration: 56.98 s (+/- 81.46 s)
Out[0]: 
  Match_ID  Tick  Timestamp_ms  ...    Opp_R18    Opp_R90    Strategy_Profile
0  M_00001     0             0  ...  -1.000000  -1.000000     JUGGERNAUT_PUSH
1  M_00001     0             0  ...  -1.000000  -1.000000  AGGRESSIVE_CHARGER
2  M_00001     1            50  ...  26.142296  -1.000000     JUGGERNAUT_PUSH
3  M_00001     1            50  ...  -1.000000  11.144795  AGGRESSIVE_CHARGER
4  M_00001     2           100  ...  11.436693  -1.000000     JUGGERNAUT_PUSH

[5 rows x 23 columns]


  Match_ID  Tick  Timestamp_ms  ...    Opp_R18    Opp_R90    Strategy_Profile
0  M_00001     0             0  ...  -1.000000  -1.000000     JUGGERNAUT_PUSH
1  M_00001     0             0  ...  -1.000000  -1.000000  AGGRESSIVE_CHARGER
2  M_00001     1            50  ...  26.142296  -1.000000     JUGGERNAUT_PUSH
3  M_00001     1            50  ...  -1.000000  11.144795  AGGRESSIVE_CHARGER
4  M_00001     2           100  ...  11.436693  -1.000000     JUGGERNAUT_PUSH

[5 rows x 23 columns]

### 3. Generate 3 kg Mega Sumo Telemetry (1,000 Matches)

In [8]:
df_mega_raw = generate_batch_telemetry(
    config=MEGA_3KG_CONFIG,
    adversary_name="RANDOM_MIX",
    num_matches=50,
    output_filename="telemetry_mega_3kg_raw.parquet",
    seed=101
)
df_mega_raw.head()

[Simulating 50 matches] Class: MEGA_3KG | Adversary: RANDOM_MIX | Formations: RANDOM_MIX vs RANDOM_MIX...
Simulating Matches: 100%|##########| 50/50 [00:17<00:00,  2.82it/s]
Saved 135,364 raw ticks across 50 matches to /Users/connorshyan/UM/SumoDigitalTwin/data/raw/telemetry_mega_3kg_raw.parquet

--- Match Outcome Summary ---
Status
BOT_A_WIN    36.0 %
BOT_B_WIN    34.0 %
DRAW         30.0 %
Name: proportion, dtype: str
Mean Match Duration: 67.63 s (+/- 81.65 s)
Out[0]: 
  Match_ID  Tick  Timestamp_ms  ... Opp_R22_5    Opp_R90   Strategy_Profile
0  M_00001     0             0  ...      -1.0  -1.000000  DEFENSIVE_SWEEPER
1  M_00001     0             0  ...      -1.0  -1.000000     RANDOM_FLANKER
2  M_00001     1            50  ...      -1.0  28.824780  DEFENSIVE_SWEEPER
3  M_00001     1            50  ...      -1.0  -1.000000     RANDOM_FLANKER
4  M_00001     2           100  ...      -1.0  31.324378  DEFENSIVE_SWEEPER

[5 rows x 25 columns]


  Match_ID  Tick  Timestamp_ms  ... Opp_R22_5    Opp_R90   Strategy_Profile
0  M_00001     0             0  ...      -1.0  -1.000000  DEFENSIVE_SWEEPER
1  M_00001     0             0  ...      -1.0  -1.000000     RANDOM_FLANKER
2  M_00001     1            50  ...      -1.0  28.824780  DEFENSIVE_SWEEPER
3  M_00001     1            50  ...      -1.0  -1.000000     RANDOM_FLANKER
4  M_00001     2           100  ...      -1.0  31.324378  DEFENSIVE_SWEEPER

[5 rows x 25 columns]

### 4. Telemetry Distribution Visualizations

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State Distribution
df_kit_raw["Current_State"].value_counts().plot(kind="bar", ax=axes[0], color="royalblue", edgecolor="black")
axes[0].set_title("1 kg Kit Sumo: State Frequency Distribution")
axes[0].set_ylabel("Tick Count")
axes[0].tick_params(axis='x', rotation=45)

# Linear Velocity Distribution
v_left = (df_kit_raw["Action_PWM_Left"] / 255.0) * KIT_1KG_CONFIG.v_max_cms
v_right = (df_kit_raw["Action_PWM_Right"] / 255.0) * KIT_1KG_CONFIG.v_max_cms
v_mean = (v_left + v_right) / 2.0
axes[1].hist(v_mean, bins=30, color="darkorange", edgecolor="black", alpha=0.8)
axes[1].set_title("1 kg Kit Sumo: Mean Linear Velocity (cm/s)")
axes[1].set_xlabel("Velocity (cm/s)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Fallback to a different backend
<ipython-input-1-efd21ca6aeb6>:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Stage 2 ETL Partitioning (Agent vs Observer Isolation)

In [12]:
from data_pipeline.etl_pipeline import process_raw_telemetry

print("--- Processing Kit 1kg Telemetry ---")
res_kit = process_raw_telemetry(df_kit_raw, weight_class="KIT_1KG")
print(f"Kit Agent Dataset: {res_kit['agent_path']} ({res_kit['cleaned_rows']:,} records)")
print(f"Kit Observer Dataset: {res_kit['observer_path']}")

print("\n--- Processing Mega 3kg Telemetry ---")
res_mega = process_raw_telemetry(df_mega_raw, weight_class="MEGA_3KG")
print(f"Mega Agent Dataset: {res_mega['agent_path']} ({res_mega['cleaned_rows']:,} records)")
print(f"Mega Observer Dataset: {res_mega['observer_path']}")


--- Processing Kit 1kg Telemetry ---
Kit Agent Dataset: /Users/connorshyan/UM/SumoDigitalTwin/data/agent/telemetry_kit_1kg_agent.parquet (114,060 records)
Kit Observer Dataset: /Users/connorshyan/UM/SumoDigitalTwin/data/observer/telemetry_kit_1kg_observer.parquet

--- Processing Mega 3kg Telemetry ---
Mega Agent Dataset: /Users/connorshyan/UM/SumoDigitalTwin/data/agent/telemetry_mega_3kg_agent.parquet (135,364 records)
Mega Observer Dataset: /Users/connorshyan/UM/SumoDigitalTwin/data/observer/telemetry_mega_3kg_observer.parquet
